# 01 数据下载

本Notebook负责下载所有需要的金融数据：
- 10只股票的日度行情数据（baostock）
- 沪深300和中证500指数数据（baostock）
- CPI和M2宏观数据（akshare）
- 财务指标数据（baostock + akshare）

In [ ]:
# 导入必要的库
import baostock as bs
import pandas as pd
import numpy as np
import os
from datetime import datetime

# 导入akshare
import akshare as ak
print("akshare 已加载")

# 设置项目根目录
PROJECT_ROOT = r"D:\2026研一下\dsfin-hw\dshw-p02"
os.chdir(PROJECT_ROOT)
print(f"当前工作目录: {os.getcwd()}")

akshare 已加载
当前工作目录: D:\2026研一下\dsfin-hw\dshw-p02


In [ ]:
# 定义股票列表
stocks = [
    {"code": "sh.600036", "name": "招商银行", "industry": "银行"},
    {"code": "sh.601328", "name": "交通银行", "industry": "银行"},
    {"code": "sz.002594", "name": "比亚迪", "industry": "汽车"},
    {"code": "sh.600104", "name": "上汽集团", "industry": "汽车"},
    {"code": "sz.000002", "name": "万科A", "industry": "房地产"},
    {"code": "sh.600048", "name": "保利发展", "industry": "房地产"},
    {"code": "sh.600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "sz.000858", "name": "五粮液", "industry": "白酒"},
    {"code": "sh.601012", "name": "隆基绿能", "industry": "能源"},
    {"code": "sh.600028", "name": "中国石化", "industry": "能源"}
]

indices = [{"code": "sh.000300", "name": "沪深300"}, {"code": "sh.000905", "name": "中证500"}]
START_DATE = "2020-01-01"
END_DATE = "2026-04-07"

In [ ]:
# 日志记录函数
def log_download(status, item_type, item_code, shape=None, error=None, source=None):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_line = f"[{timestamp}] {status:8s} {item_type}_{item_code}"
    if shape:
        log_line += f"  shape={shape}"
    if source:
        log_line += f"  source={source}"
    if error:
        log_line += f"  Error: {error}"
    
    log_path = os.path.join(PROJECT_ROOT, "download_log.txt")
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(log_line + "\n")
    print(log_line)

# 确保目录存在
for dir_name in ["data/stock", "data/index", "data/macro", "data/finance", "data/clean", "data/combined", "output"]:
    dir_path = os.path.join(PROJECT_ROOT, dir_name.replace('/', os.sep))
    os.makedirs(dir_path, exist_ok=True)
    print(f"确保目录存在: {dir_path}")

# 清空旧日志
log_path = os.path.join(PROJECT_ROOT, "download_log.txt")
if os.path.exists(log_path):
    os.remove(log_path)

确保目录存在: D:\2026研一下\dsfin-hw\dshw-p02\data\stock
确保目录存在: D:\2026研一下\dsfin-hw\dshw-p02\data\index
确保目录存在: D:\2026研一下\dsfin-hw\dshw-p02\data\macro
确保目录存在: D:\2026研一下\dsfin-hw\dshw-p02\data\finance
确保目录存在: D:\2026研一下\dsfin-hw\dshw-p02\data\clean
确保目录存在: D:\2026研一下\dsfin-hw\dshw-p02\data\combined
确保目录存在: D:\2026研一下\dsfin-hw\dshw-p02\output


## 1. 下载股票行情数据 (baostock)

In [ ]:
# 登录baostock
lg = bs.login()
print(f"baostock登录结果: {lg.error_msg}")

def download_stock_data(stock_code, stock_name):
    """下载单只股票的后复权日度行情数据"""
    try:
        rs = bs.query_history_k_data_plus(
            stock_code, "date,open,high,low,close,volume,amount",
            start_date=START_DATE, end_date=END_DATE, frequency="d", adjustflag="2"
        )
        
        data_list = []
        while (rs.error_code == '0') & rs.next():
            data_list.append(rs.get_row_data())
        
        if data_list:
            df = pd.DataFrame(data_list, columns=rs.fields)
            for col in ['open', 'high', 'low', 'close', 'volume', 'amount']:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            
            code_simple = stock_code.replace('.', '_')
            filepath = os.path.join(PROJECT_ROOT, "data", "stock", f"stock_{code_simple}.csv")
            df.to_csv(filepath, index=False, encoding="utf-8-sig")
            log_download("SUCCESS", "stock", code_simple, shape=df.shape, source="baostock")
            return df
        else:
            log_download("FAILED", "stock", stock_code.replace('.', '_'), error="No data returned", source="baostock")
    except Exception as e:
        log_download("FAILED", "stock", stock_code.replace('.', '_'), error=str(e), source="baostock")
    return None

# 下载所有股票
stock_data = {}
for stock in stocks:
    df = download_stock_data(stock['code'], stock['name'])
    if df is not None:
        stock_data[stock['code']] = df

login success!
baostock登录结果: success
[2026-04-08 09:33:30] SUCCESS  stock_sh_600036  shape=(1515, 7)  source=baostock
[2026-04-08 09:33:32] SUCCESS  stock_sh_601328  shape=(1515, 7)  source=baostock
[2026-04-08 09:33:38] SUCCESS  stock_sz_002594  shape=(1515, 7)  source=baostock
[2026-04-08 09:33:41] SUCCESS  stock_sh_600104  shape=(1515, 7)  source=baostock
[2026-04-08 09:33:45] SUCCESS  stock_sz_000002  shape=(1515, 7)  source=baostock
[2026-04-08 09:33:47] SUCCESS  stock_sh_600048  shape=(1515, 7)  source=baostock
[2026-04-08 09:33:50] SUCCESS  stock_sh_600519  shape=(1515, 7)  source=baostock
[2026-04-08 09:33:53] SUCCESS  stock_sz_000858  shape=(1515, 7)  source=baostock
[2026-04-08 09:33:58] SUCCESS  stock_sh_601012  shape=(1515, 7)  source=baostock
[2026-04-08 09:33:59] SUCCESS  stock_sh_600028  shape=(1515, 7)  source=baostock


## 2. 下载指数数据 (baostock)

In [ ]:
def download_index_data(index_code, index_name):
    try:
        rs = bs.query_history_k_data_plus(
            index_code, "date,open,high,low,close,volume,amount",
            start_date=START_DATE, end_date=END_DATE, frequency="d", adjustflag="3"
        )
        
        data_list = []
        while (rs.error_code == '0') & rs.next():
            data_list.append(rs.get_row_data())
        
        if data_list:
            df = pd.DataFrame(data_list, columns=rs.fields)
            for col in ['open', 'high', 'low', 'close', 'volume', 'amount']:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            
            code_simple = index_code.replace('.', '_')
            filepath = os.path.join(PROJECT_ROOT, "data", "index", f"index_{code_simple}.csv")
            df.to_csv(filepath, index=False, encoding="utf-8-sig")
            log_download("SUCCESS", "index", code_simple, shape=df.shape, source="baostock")
            return df
        else:
            log_download("FAILED", "index", index_code.replace('.', '_'), error="No data returned", source="baostock")
    except Exception as e:
        log_download("FAILED", "index", index_code.replace('.', '_'), error=str(e), source="baostock")
    return None

# 下载所有指数
index_data = {}
for index in indices:
    df = download_index_data(index['code'], index['name'])
    if df is not None:
        index_data[index['code']] = df

[2026-04-08 09:34:02] SUCCESS  index_sh_000300  shape=(1515, 7)  source=baostock
[2026-04-08 09:34:04] SUCCESS  index_sh_000905  shape=(1515, 7)  source=baostock


## 3. 下载宏观经济数据 (akshare)

In [ ]:
def download_macro_data():
    """从akshare下载CPI和M2宏观数据"""
    
    # === CPI数据 ===
    print("正在从akshare下载CPI数据...")
    cpi_df = None
    try:
        # 尝试多种akshare CPI接口
        try:
            cpi_raw = ak.macro_china_cpi()
        except:
            try:
                cpi_raw = ak.macro_china_cpi_monthly()
            except:
                cpi_raw = None
        
        if cpi_raw is not None and len(cpi_raw) > 0:
            print(f"akshare CPI数据列: {list(cpi_raw.columns)}")
            print(f"CPI数据前3行:\n{cpi_raw.head(3)}")
            
            # 自动识别日期和值列
            date_col = None
            value_col = None
            
            for col in cpi_raw.columns:
                col_lower = str(col).lower()
                if '日期' in str(col) or 'date' in col_lower or '时间' in str(col) or '月份' in str(col) or '年' in str(col):
                    date_col = col
                if '值' in str(col) or 'value' in col_lower or '同比' in str(col) or 'cpi' in col_lower or '涨' in str(col) or '%' in str(col):
                    value_col = col
            
            if date_col and value_col:
                print(f"识别到: 日期列={date_col}, 值列={value_col}")
                cpi_df = pd.DataFrame({
                    'date': pd.to_datetime(cpi_raw[date_col]),
                    'indicator': 'cpi',
                    'value': pd.to_numeric(cpi_raw[value_col], errors='coerce')
                })
                cpi_df['date'] = cpi_df['date'].dt.strftime('%Y-%m-%d')
                cpi_df = cpi_df[cpi_df['date'] >= '2020-01-01'].sort_values('date').reset_index(drop=True)
        
        if cpi_df is not None and len(cpi_df) > 0:
            cpi_path = os.path.join(PROJECT_ROOT, "data", "macro", "macro_cpi.csv")
            cpi_df.to_csv(cpi_path, index=False, encoding="utf-8-sig")
            log_download("SUCCESS", "macro", "cpi", shape=cpi_df.shape, source="akshare")
        else:
            log_download("FAILED", "macro", "cpi", error="Could not process CPI data", source="akshare")
    except Exception as e:
        log_download("FAILED", "macro", "cpi", error=str(e), source="akshare")
        cpi_df = None
    
    # === M2数据 ===
    print("正在从akshare下载M2数据...")
    m2_df = None
    try:
        # 尝试多种akshare M2接口
        try:
            m2_raw = ak.macro_china_m2()
        except:
            try:
                m2_raw = ak.macro_china_m2_monthly()
            except:
                m2_raw = None
        
        if m2_raw is not None and len(m2_raw) > 0:
            print(f"akshare M2数据列: {list(m2_raw.columns)}")
            
            # 自动识别日期和值列
            date_col = None
            value_col = None
            
            for col in m2_raw.columns:
                col_lower = str(col).lower()
                if '日期' in str(col) or 'date' in col_lower or '时间' in str(col) or '月份' in str(col) or '年' in str(col):
                    date_col = col
                if '值' in str(col) or 'value' in col_lower or '同比' in str(col) or 'm2' in col_lower or '增长' in str(col) or '%' in str(col):
                    value_col = col
            
            if date_col and value_col:
                print(f"识别到: 日期列={date_col}, 值列={value_col}")
                m2_df = pd.DataFrame({
                    'date': pd.to_datetime(m2_raw[date_col]),
                    'indicator': 'm2',
                    'value': pd.to_numeric(m2_raw[value_col], errors='coerce')
                })
                m2_df['date'] = m2_df['date'].dt.strftime('%Y-%m-%d')
                m2_df = m2_df[m2_df['date'] >= '2020-01-01'].sort_values('date').reset_index(drop=True)
        
        if m2_df is not None and len(m2_df) > 0:
            m2_path = os.path.join(PROJECT_ROOT, "data", "macro", "macro_m2.csv")
            m2_df.to_csv(m2_path, index=False, encoding="utf-8-sig")
            log_download("SUCCESS", "macro", "m2", shape=m2_df.shape, source="akshare")
        else:
            log_download("FAILED", "macro", "m2", error="Could not process M2 data", source="akshare")
    except Exception as e:
        log_download("FAILED", "macro", "m2", error=str(e), source="akshare")
        m2_df = None
    
    return cpi_df, m2_df

cpi_data, m2_data = download_macro_data()

正在从akshare下载CPI数据...
akshare CPI数据列: ['月份', '全国-当月', '全国-同比增长', '全国-环比增长', '全国-累计', '城市-当月', '城市-同比增长', '城市-环比增长', '城市-累计', '农村-当月', '农村-同比增长', '农村-环比增长', '农村-累计']
CPI数据前3行:
          月份  全国-当月  全国-同比增长  全国-环比增长  全国-累计  城市-当月  城市-同比增长  城市-环比增长  城市-累计  \
0  2026年02月份  101.3      1.3      1.0  100.8  101.4      1.4      1.0  100.8   
1  2026年01月份  100.2      0.2      0.2  100.2  100.2      0.2      0.2  100.2   
2  2025年12月份  100.8      0.8      0.2  100.0  100.9      0.9      0.2  100.1   

   农村-当月  农村-同比增长  农村-环比增长  农村-累计  
0  100.9      0.9      0.7  100.5  
1  100.1      0.1      0.2  100.1  
2  100.6      0.6      0.2   99.8  
识别到: 日期列=月份, 值列=农村-同比增长
[2026-04-08 09:34:04] FAILED   macro_cpi  source=akshare  Error: Unknown datetime string format, unable to parse: 2026年02月份, at position 0
正在从akshare下载M2数据...
[2026-04-08 09:34:04] FAILED   macro_m2  source=akshare  Error: Could not process M2 data


C:\Users\AURORA-鱼\AppData\Local\Temp\ipykernel_9924\988674566.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  'date': pd.to_datetime(cpi_raw[date_col]),


## 4. 下载财务指标数据 (baostock + akshare)

In [ ]:
def download_financial_data():
    """从baostock下载财务指标数据（ROE和净利润率）"""
    
    finance_records = []
    
    for stock in stocks:
        code = stock['code']
        print(f"正在下载 {stock['name']} 财务数据...")
        try:
            for year in range(2020, 2025):
                rs = bs.query_profit_data(code=code, year=year, quarter=4)
                data_list = []
                while (rs.error_code == '0') & rs.next():
                    data_list.append(rs.get_row_data())
                
                if data_list:
                    df_profit = pd.DataFrame(data_list, columns=rs.fields)
                    # ROE
                    if 'avgroe' in df_profit.columns:
                        roe = pd.to_numeric(df_profit['avgroe'].iloc[0], errors='coerce')
                        if not pd.isna(roe):
                            finance_records.append({
                                'code': code,
                                'year': year,
                                'indicator': 'roe',
                                'value': roe
                            })
                    # 净利润率
                    if 'netprofit' in df_profit.columns and 'operatingrevenue' in df_profit.columns:
                        net_profit = pd.to_numeric(df_profit['netprofit'].iloc[0], errors='coerce')
                        revenue = pd.to_numeric(df_profit['operatingrevenue'].iloc[0], errors='coerce')
                        if not pd.isna(net_profit) and not pd.isna(revenue) and revenue != 0:
                            net_margin = net_profit / revenue * 100
                            finance_records.append({
                                'code': code,
                                'year': year,
                                'indicator': 'net_margin',
                                'value': net_margin
                            })
        except Exception as e:
            print(f"{stock['name']} {year}年财务数据下载失败: {e}")
    
    if finance_records:
        finance_df = pd.DataFrame(finance_records)
        filepath = os.path.join(PROJECT_ROOT, "data", "finance", "finance_ratios.csv")
        finance_df.to_csv(filepath, index=False, encoding="utf-8-sig")
        log_download("SUCCESS", "finance", "ratios", shape=finance_df.shape, source="baostock")
        return finance_df
    else:
        log_download("FAILED", "finance", "ratios", error="No financial data retrieved", source="baostock")
        return None

finance_data = download_financial_data()

正在下载 招商银行 财务数据...
正在下载 交通银行 财务数据...
正在下载 比亚迪 财务数据...
正在下载 上汽集团 财务数据...
正在下载 万科A 财务数据...
正在下载 保利发展 财务数据...
正在下载 贵州茅台 财务数据...
正在下载 五粮液 财务数据...
正在下载 隆基绿能 财务数据...
正在下载 中国石化 财务数据...
[2026-04-08 09:35:07] FAILED   finance_ratios  source=baostock  Error: No financial data retrieved


In [ ]:
# 退出baostock
bs.logout()
print("\n" + "="*60)
print("数据下载完成！")
print("="*60)

# 检查已下载的文件
print("\n已下载文件统计:")
for dir_name in ["data/stock", "data/index", "data/macro", "data/finance"]:
    dir_path = os.path.join(PROJECT_ROOT, dir_name.replace('/', os.sep))
    if os.path.exists(dir_path):
        files = [f for f in os.listdir(dir_path) if f.endswith('.csv')]
        print(f"  {dir_name}: {len(files)} 个文件")

logout success!

数据下载完成！

已下载文件统计:
  data/stock: 10 个文件
  data/index: 2 个文件
  data/macro: 0 个文件
  data/finance: 1 个文件
